In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, GRU, SimpleRNN
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('inshorts.csv')
df.head()

,Category,Article
0,Technology,tv future in the hands of viewers with home th...
1,Business,worldcom boss left books alone former worldc...
2,Sports,tigers wary of farrell gamble leicester say ...
3,Sports,yeading face newcastle in fa cup premiership s...
4,Entertainment,ocean s twelve raids box office ocean s twelve...


In [3]:
df.shape

(2225, 2)

In [4]:
max_features = 5000
maxlen = 500

embedding_size = 100
batch_size = 512
epochs = 10

In [5]:
def preprocess_text(df, text_column):
  df[text_column] = df[text_column].apply(lambda x: x.lower())
  return df

df = preprocess_text(df, 'Article')

In [6]:
tokenizer = Tokenizer(num_words = max_features)
tokenizer.fit_on_texts(df['Article'])

sequences = tokenizer.texts_to_sequences(df['Article'])
data = pad_sequences(sequences, maxlen = maxlen)

In [7]:
data.shape

(2225, 500)

In [8]:
le = LabelEncoder()

labels = le.fit_transform(df['Category'])
labels = tf.keras.utils.to_categorical(labels)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size = 0.2, random_state=42)

In [11]:
def load_glove_matrix(path, tok, max_feats, dim):
    embedding_index = {}
    
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embedding_index[word] = coefs
    
    vocab_size = min(max_feats, len(tok.word_index)) #+ 1  # +1 for padding idx=0
    matrix = np.zeros((vocab_size, dim), dtype="float32")

    for word, idx in tok.word_index.items():
        if idx >= vocab_size:
            continue
        vec = embedding_index.get(word)
        if vec is not None:
            matrix[idx] = vec
    print(f"Embedding matrix shape: {matrix.shape}")
    return matrix

embedding_matrix = load_glove_matrix('glove.6B.100d.txt', tokenizer, max_features, embedding_size)

Embedding matrix shape: (5000, 100)


In [12]:
model = Sequential([
    Embedding(max_features, embedding_size, weights=[embedding_matrix], input_length=maxlen, trainable=False),
    SimpleRNN(50), # LSTM(50) 
    Dense(len(np.unique(df['Category'])), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


C:\Users\arunk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=5)

# Model training
model.fit(X_train, y_train,
batch_size=batch_size, epochs=epochs, validation_data=(X_test, y_test), verbose=2,
callbacks=[early_stopping])

Epoch 1/10
4/4 - 2s - 585ms/step - accuracy: 0.1860 - loss: 1.8166 - val_accuracy: 0.2135 - val_loss: 1.7018
Epoch 2/10
4/4 - 1s - 204ms/step - accuracy: 0.2416 - loss: 1.6530 - val_accuracy: 0.2404 - val_loss: 1.6557
Epoch 3/10
4/4 - 2s - 394ms/step - accuracy: 0.2966 - loss: 1.6044 - val_accuracy: 0.2697 - val_loss: 1.6219
Epoch 4/10
4/4 - 1s - 307ms/step - accuracy: 0.3146 - loss: 1.5602 - val_accuracy: 0.2921 - val_loss: 1.5779
Epoch 5/10
4/4 - 1s - 333ms/step - accuracy: 0.3506 - loss: 1.5125 - val_accuracy: 0.3191 - val_loss: 1.5412
Epoch 6/10
4/4 - 1s - 274ms/step - accuracy: 0.3770 - loss: 1.4766 - val_accuracy: 0.3348 - val_loss: 1.5181
Epoch 7/10
4/4 - 1s - 287ms/step - accuracy: 0.3888 - loss: 1.4473 - val_accuracy: 0.3371 - val_loss: 1.4981
Epoch 8/10
4/4 - 1s - 338ms/step - accuracy: 0.4135 - loss: 1.4164 - val_accuracy: 0.3416 - val_loss: 1.4781
Epoch 9/10
4/4 - 1s - 283ms/step - accuracy: 0.4281 - loss: 1.3859 - val_accuracy: 0.3573 - val_loss: 1.4577
Epoch 10/10
4/4 - 1

In [14]:
def predict_category(text, tokenizer, model, label_encoder, max_len):

  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])

  padded_seq = pad_sequences(seq, maxlen = max_len)

  pred= model.predict(padded_seq)

  pred_label_index = np.argmax(pred, axis=1)

  pred_label = label_encoder.inverse_transform(pred_label_index)

  return pred_label[0]

In [15]:
input_text = 'I need to create better algorithm for predicting the stock market'

predicted_category = predict_category(input_text, tokenizer, model, le, maxlen)

print("predicted category: ", predicted_category)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
predicted category:  Business


In [16]:
input_text = 'I love playing football, I make several goals'

predicted_category = predict_category(input_text, tokenizer, model, le, maxlen)

print("predicted category: ", predicted_category)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
predicted category:  Sports
